In [70]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))


In [71]:
import pandas as pd
import numpy as np
from config import DATA_DIR
from config import PROCESSED_DATA_DIR


DATA_PATH = DATA_DIR / "raw" / "Protest_pg_ready.csv"
df = pd.read_csv(DATA_PATH, sep=";")


In [72]:
def map_actor_type(text):
    if pd.isna(text):
        return "other"
    text = text.lower()

    # ایجاد یک لیست از کلمات کلیدی برای هر گروه
    student_keywords = ["student", "youth"]
    labor_keywords = ["labor", "worker", "unions", "employee"]
    political_keywords = ["party", "political", "government", "activist"]
    identity_keywords = ["ethnic", "religion", "identity", "community"]
    professional_keywords = ["professional"]
    mass_public_keywords = ["citizen", "public", "people", "crowd"]

    # بررسی وجود کلمات کلیدی
    if any(k in text for k in student_keywords):
        return "student_group"
    elif any(k in text for k in labor_keywords):
        return "labor_group"
    elif any(k in text for k in political_keywords):
        return "political_group"
    elif any(k in text for k in identity_keywords):
        return "identity_group"
    elif any(k in text for k in professional_keywords):
        return "professional_group"
    elif any(k in text for k in mass_public_keywords):
        return "mass_public"
    else:
        return "other"



In [73]:
demand_cols = [
    "Primary Protester Demands [Demand 1]",
    "Primary Protester Demands [Demand 2]",
    "Primary Protester Demands [Demand 3]",
    "Primary Protester Demands [Demand 4]"
]

def map_demand_type(row):
    text = " ".join([str(row[c]).lower() for c in demand_cols if pd.notna(row[c])])
    if text == "":
        return "no_demand"
    if any(k in text for k in ["election", "government", "policy", "resign"]):
        return "political_demand"
    if any(k in text for k in ["wage", "price", "economic", "fuel"]):
        return "economic_demand"
    if any(k in text for k in ["freedom", "speech", "rights"]):
        return "freedom_demand"
    if any(k in text for k in ["ethnic", "relig", "identity"]):
        return "identity_demand"
    return "other_demand"

df["Demand_Type"] = df.apply(map_demand_type, axis=1)


In [74]:
response_cols = [
    "Primary State Response to protests [Response 1]",
    "Primary State Response to protests [Response 2?¾]",
    "Primary State Response to protests [Response 3]",
    "Primary State Response to protests [Response 4]",
    "Primary State Response to protests [Response 5]",
    "Primary State Response to protests [Response 6]",
    "Primary State Response to protests [Response 7]"
]

STATE_RESPONSE_MAP = {
    "ignore": 0,
    "accomodation": 1,
    "crowd dispersal": 2,
    "arrests": 3,
    "beatings": 4,
    "shootings": 5,
    "killings": 6
}

def compute_rsi(row):
    scores = []
    for col in response_cols:
        if col in row and pd.notna(row[col]):
            val = str(row[col]).strip().lower()
            if val in STATE_RESPONSE_MAP:
                scores.append(STATE_RESPONSE_MAP[val])
    return max(scores) if scores else 0

df["RSI"] = df.apply(compute_rsi, axis=1)


In [75]:
df["Actor_Type"] = df["Protester Group Identity"].apply(map_actor_type)


In [76]:
df["RSI"].value_counts().sort_index()


RSI
0    8744
1     902
2    2300
3    1344
4     598
5     385
6     700
Name: count, dtype: int64

In [77]:
final_cols = [
    "Country name",
    "Demand_Type",
    "RSI"
]

events_clean = df[final_cols].dropna(subset=["RSI"])


In [78]:
events_clean.shape
events_clean["RSI"].value_counts().sort_index()


RSI
0    8744
1     902
2    2300
3    1344
4     598
5     385
6     700
Name: count, dtype: int64

In [79]:
events_clean["Demand_Type"].value_counts()


Demand_Type
other_demand        10109
economic_demand      1813
no_demand            1757
political_demand     1294
Name: count, dtype: int64

In [80]:
DEMAND_MAPPING = {
    "political behavior, process": "political_demand",
    "removal of politician": "political_demand",
    "police brutality": "political_demand",
    "social restrictions": "political_demand",

    "labor wage dispute": "economic_demand",
    "price increases, tax policy": "economic_demand",

    "land farm issue": "land farm issue"
}


In [81]:
demand_cols = [
    'Primary Protester Demands [Demand 1]',
    'Primary Protester Demands [Demand 2]',
    'Primary Protester Demands [Demand 3]',
    'Primary Protester Demands [Demand 4]'
]

def map_event_demand(row):
    demands = row[demand_cols].dropna().astype(str).str.strip().tolist()

    mapped = [DEMAND_MAPPING.get(d) for d in demands if d in DEMAND_MAPPING]

    if "political_demand" in mapped:
        return "political_demand"
    if "economic_demand" in mapped:
        return "economic_demand"
    if mapped:
        return mapped[0]
    return "no_demand"


In [82]:
df["Demand_Type"] = df.apply(map_event_demand, axis=1)


In [83]:
df["Demand_Type"].value_counts()


Demand_Type
political_demand    10894
economic_demand      2066
no_demand            1757
land farm issue       256
Name: count, dtype: int64

In [84]:
df[df["Demand_Type"] == "political_demand"][demand_cols].head(10)


,Primary Protester Demands [Demand 1],Primary Protester Demands [Demand 2],Primary Protester Demands [Demand 3],Primary Protester Demands [Demand 4]
0,"political behavior, process",labor wage dispute,NaN,NaN
1,"political behavior, process",NaN,NaN,NaN
2,"political behavior, process",NaN,NaN,NaN
4,"political behavior, process",NaN,NaN,NaN
5,police brutality,NaN,NaN,NaN
8,police brutality,NaN,NaN,NaN
9,"political behavior, process",NaN,NaN,NaN
10,"political behavior, process",NaN,NaN,NaN
11,"political behavior, process",NaN,NaN,NaN
16,"political behavior, process",labor wage dispute,NaN,NaN


In [85]:
events_clean.head()
events_clean.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14973 entries, 0 to 14972
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Country name  14973 non-null  object
 1   Demand_Type   14973 non-null  object
 2   RSI           14973 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 351.1+ KB
